# Weather Data Check

ตรวจสอบและเตรียมข้อมูลปริมาณฝนรายเดือนรายจังหวัด
สำหรับนำไปเชื่อมกับข้อมูลนักท่องเที่ยวอุทยานแห่งชาติ

In [1]:
import pandas as pd

rain = pd.read_csv("../data/rainfall_province_monthly.csv")

rain.head()

,YEAR,MONTH,PROV_ID,PROV_T,MinRain,MaxRain,AvgRain
0,2018,1,10,กรุงเทพมหานคร,54.299999,257.230011,142.119137
1,2018,1,11,สมุทรปราการ,76.250000,256.100006,137.302046
2,2018,1,12,นนทบุรี,38.360001,161.470001,113.433771
3,2018,1,13,ปทุมธานี,51.439999,116.500000,82.901688
4,2018,1,14,พระนครศรีอยุธยา,8.850000,88.589996,39.960089


1. สำรวจโครงสร้างข้อมูลฝน

In [2]:
print("ขนาดข้อมูล:", rain.shape)

print("\nชื่อคอลัมน์:")
print(rain.columns.tolist())

ขนาดข้อมูล: (7931, 7)

ชื่อคอลัมน์:
['YEAR', 'MONTH', 'PROV_ID', 'PROV_T', 'MinRain', 'MaxRain', 'AvgRain']


2. ตรวจคุณภาพและความครอบคลุมของข้อมูลฝน
ตรวจชนิดข้อมูล Missing Values ข้อมูลซ้ำ ช่วงปี เดือน และจำนวนจังหวัด
เพื่อประเมินความพร้อมของ Dataset ก่อนนำไปทำความสะอาดและเชื่อมกับข้อมูลอุทยาน

In [ ]:
print("ชนิดข้อมูล:")
print(rain.dtypes)

print("\nMissing Values:")
print(rain.isna().sum())

print("\nข้อมูลซ้ำ จังหวัด + ปี + เดือน:")
print(
    rain.duplicated(
        subset=["YEAR", "MONTH", "PROV_ID"]
    ).sum()
)

print("\nช่วงปี:")
print(rain["YEAR"].min(), "-", rain["YEAR"].max())

print("\nเดือนที่มี:")
print(sorted(rain["MONTH"].unique()))

print("\nจำนวนจังหวัด:")
print(rain["PROV_ID"].nunique())

3. ตรวจความครบถ้วนและความสมเหตุสมผลของข้อมูลฝน

ตรวจจำนวนข้อมูลในแต่ละปีว่าครบ 12 เดือนและ 77 จังหวัดหรือไม่
รวมถึงตรวจค่าปริมาณฝนที่ผิดปกติ เช่น ค่าติดลบ หรือค่า Min/Avg/Max ที่ไม่สัมพันธ์กัน

In [ ]:
# จำนวน record ในแต่ละปี
records_by_year = rain.groupby("YEAR").size()

print("จำนวนข้อมูลในแต่ละปี:")
print(records_by_year)

print("\nจำนวนเดือนในแต่ละปี:")
print(rain.groupby("YEAR")["MONTH"].nunique())

print("\nจำนวนจังหวัดในแต่ละปี:")
print(rain.groupby("YEAR")["PROV_ID"].nunique())

# ตรวจค่าฝนติดลบ
print("\nจำนวนค่าฝนติดลบ:")
print(
    (rain[["MinRain", "MaxRain", "AvgRain"]] < 0).sum()
)

# ตรวจความสัมพันธ์ Min <= Avg <= Max
invalid_rain = rain[
    (rain["MinRain"] > rain["AvgRain"]) |
    (rain["AvgRain"] > rain["MaxRain"])
]

print("\nจำนวนแถวที่ Min/Avg/Max ไม่สมเหตุสมผล:")
print(len(invalid_rain))

In [8]:
# เลือกเฉพาะปีที่มีข้อมูลครบ 12 เดือน
complete_years = (
    rain.groupby("YEAR")["MONTH"]
    .nunique()
)

complete_years = complete_years[
    complete_years == 12
].index

rain_complete = rain[
    rain["YEAR"].isin(complete_years)
].copy()

print("ปีที่ใช้สำหรับการวิเคราะห์เต็มปี:")
print(complete_years.tolist())

print("\nขนาดข้อมูล:")
print(rain_complete.shape)

ปีที่ใช้สำหรับการวิเคราะห์เต็มปี:
[2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

ขนาดข้อมูล:
(7392, 7)


4. สร้าง Weather Clean Dataset

ปรับชื่อคอลัมน์และสร้างวันที่มาตรฐานจากปีและเดือน
เพื่อเตรียมข้อมูลปริมาณฝนสำหรับเชื่อมกับข้อมูลนักท่องเที่ยว
และนำไปใช้ในฐานข้อมูลและการวิเคราะห์ต่อไป

In [9]:
weather_clean = rain_complete.rename(columns={
    "YEAR": "year",
    "MONTH": "month",
    "PROV_ID": "province_id",
    "PROV_T": "province",
    "MinRain": "min_rain",
    "MaxRain": "max_rain",
    "AvgRain": "avg_rain"
}).copy()

# สร้างวันที่แทนแต่ละเดือน
weather_clean["date"] = pd.to_datetime(
    dict(
        year=weather_clean["year"],
        month=weather_clean["month"],
        day=1
    )
)

# เรียงตามวันที่และจังหวัด
weather_clean = (
    weather_clean
    .sort_values(["date", "province"])
    .reset_index(drop=True)
)

weather_clean.head()

,year,month,province_id,province,min_rain,max_rain,avg_rain,date
0,2018,1,81,กระบี่,136.830002,368.529999,261.255659,2018-01-01
1,2018,1,10,กรุงเทพมหานคร,54.299999,257.230011,142.119137,2018-01-01
2,2018,1,71,กาญจนบุรี,0.000000,21.170000,7.993726,2018-01-01
3,2018,1,46,กาฬสินธุ์,0.000000,2.220000,0.494789,2018-01-01
4,2018,1,62,กำแพงเพชร,5.310000,25.309999,10.067432,2018-01-01


In [10]:
#เซฟไฟล์
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

weather_clean.to_csv(
    processed_dir / "weather_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print(weather_clean.shape)

(7392, 8)
